# Full Pipeline Training Job

This notebook triggers the full pipeline SageMaker training job that:
1. Downloads images from URLs in manifest
2. Uploads images to S3
3. Trains the model
4. Deploys to endpoint (optional)

## Prerequisites

1. Source code uploaded to S3 (run `backend/aws/upload_source_to_s3.sh`)
2. Manifest file uploaded to S3
3. SageMaker execution role configured


In [ ]:
# Install dependencies (run once)
!pip install boto3 sagemaker aiohttp aiofiles torch torchvision pillow scikit-learn


In [ ]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch
from datetime import datetime
import json

# Configuration
region = 'us-east-1'  # Change to your region
bucket = 'your-sagemaker-bucket'  # Replace with your bucket name

# Get SageMaker session and role
boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
role = sagemaker.get_execution_role()  # Automatically gets the role

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")


In [ ]:
# Create PyTorch estimator
manifest_path = f's3://{bucket}/album-classifier/data/releases_manifest_enriched.jsonl'

estimator = PyTorch(
    entry_point='full_pipeline_train.py',
    source_dir=f's3://{bucket}/album-classifier/source',
    role=role,
    instance_type='ml.p3.2xlarge',
    instance_count=1,
    framework_version='2.5.1',
    py_version='py311',
    hyperparameters={
        'epochs': '10',
        'batch-size': '16',
        'learning-rate': '0.001',
        'manifest-path': manifest_path,
        'max-concurrent': '10',
        'skip-download': 'false',
        'skip-upload': 'false',
        'skip-training': 'false',
        'skip-deploy': 'false',
        'endpoint-instance-type': 'ml.m5.large',
    },
    output_path=f's3://{bucket}/album-classifier/models',
    code_location=f's3://{bucket}/album-classifier/code',
    sagemaker_session=sagemaker_session,
    base_job_name='album-classifier-full',
    max_run=86400 * 2,
    use_spot_instances=True,
    max_wait=86400 * 2,
    checkpoint_s3_uri=f's3://{bucket}/album-classifier/checkpoints',
    volume_size=100,
)

print("Estimator created successfully")


In [ ]:
# Start training job
print("=" * 60)
print("Starting Full Pipeline Training Job")
print("=" * 60)
print(f"Manifest: {manifest_path}")
print(f"Instance: ml.p3.2xlarge")
print(f"Output: s3://{bucket}/album-classifier/models")

estimator.fit({'training': f's3://{bucket}/album-classifier/data'})


In [ ]:
# Get job information
job_name = estimator.latest_training_job.name
print(f"Training Job: {job_name}")
print(f"Model Data: {estimator.model_data}")
print(f"\nView in console:")
print(f"https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/training-jobs/{job_name}")
